# Step 3 - Evaluate PyTorch model and report per-class accuracy


In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from pathlib import Path


REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent


DATA_PREP = REPO_ROOT / "data" / "prepared"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


# Filenames (edit ONLY these if your filenames differ)
data_file = DATA_PREP / "S1_point_all_10d_10m_20180101-20180731_Stratum1_VV-VH.csv"
classes_file = DATA_PREP / "LUCAS_2018_Copernicus_attributes_cropmap_level1-2_FROM_EXPORTS.csv"
model_path = MODELS_DIR / "MLP_torch_model.pt"


# Friendly errors if files are missing
assert data_file.exists(), f"Missing dataset file: {data_file}"
assert classes_file.exists(), f"Missing classes file: {classes_file}"
assert model_path.exists(), f"Missing model checkpoint: {model_path}"

print("Using data_file:", data_file)
print("Using classes_file:", classes_file)
print("Using model checkpoint:", model_path)


Using data_file: /workspace/eucropmap-reprod-kz/data/prepared/S1_point_all_10d_10m_20180101-20180731_Stratum1_VV-VH.csv
Using classes_file: /workspace/eucropmap-reprod-kz/data/prepared/LUCAS_2018_Copernicus_attributes_cropmap_level1-2_FROM_EXPORTS.csv
Using model checkpoint: /workspace/eucropmap-reprod-kz/models/MLP_torch_model.pt


In [2]:
class_table = pd.read_csv(classes_file)

classes_L2 = class_table["level_2"].dropna().astype(int).unique().tolist()

# official Level-2 crop classes
L2_official = {211, 212, 213, 214, 215, 216, 217, 218, 219, 221, 222, 223, 230, 231, 232, 233, 240, 250, 290}
classes_L2 = [c for c in classes_L2 if c in L2_official]

df = pd.read_csv(data_file, dtype={"level_1": int, "level_2": int})
df["Classif"] = df["level_2"]
if classes_L2:
    df = df[df["Classif"].isin(classes_L2)]

feature_regex = r'(((?<![\w\d])VH_)|((?<![\w\d])VV_))'
feature_regex += r'(2018(0[1-7]))'

X = df.filter(regex=feature_regex)
y = df["Classif"]

print(f"Prepared features shape: {X.shape}")
print(f"Prepared targets shape: {y.shape}")


Prepared features shape: (604610, 42)
Prepared targets shape: (604610,)


In [3]:
# Recreate the validation split used in training
_, X_val, _, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Load checkpoint + move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(model_path, map_location=device, weights_only=False)

input_dim = int(ckpt["input_dim"])
num_classes = int(ckpt["num_classes"])

model = nn.Sequential(
    nn.Linear(input_dim, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.BatchNorm1d(128),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, num_classes),
).to(device)

model.load_state_dict(ckpt["state_dict"])
model.eval()

class_to_idx = {int(k): int(v) for k, v in ckpt["class_to_idx"].items()}
idx_to_class = {int(k): int(v) for k, v in ckpt["idx_to_class"].items()}

scaler_mean = np.asarray(ckpt["scaler_mean"], dtype=np.float32)
scaler_scale = np.asarray(ckpt["scaler_scale"], dtype=np.float32)
scaler_scale = np.where(scaler_scale == 0, 1.0, scaler_scale)

print(f"Inference device: {device}")
print(f"Validation set size: {len(X_val)}")


Inference device: cuda
Validation set size: 151153


In [4]:
# Apply saved scaling and run batched inference
X_val_np = X_val.to_numpy(dtype=np.float32)
X_val_scaled = (X_val_np - scaler_mean) / scaler_scale
X_val_tensor = torch.from_numpy(X_val_scaled)

y_true = y_val.to_numpy(dtype=np.int64)
y_true_idx = np.array([class_to_idx[int(c)] for c in y_true], dtype=np.int64)

y_pred_idx = []
batch_size = 4096

with torch.no_grad():
    for start in range(0, len(X_val_tensor), batch_size):
        xb = X_val_tensor[start:start + batch_size].to(device, non_blocking=True)
        preds = model(xb).argmax(dim=1).cpu().numpy()
        y_pred_idx.extend(preds.tolist())

y_pred_idx = np.asarray(y_pred_idx, dtype=np.int64)
y_pred = np.array([idx_to_class[i] for i in y_pred_idx], dtype=np.int64)

overall_acc = (y_pred == y_true).mean()
print(f"Overall validation accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")


Overall validation accuracy: 0.8287 (82.87%)


In [5]:
# Per-class accuracy = correct predictions for class / total samples of class
rows = []
for class_id in sorted(np.unique(y_true)):
    mask = y_true == class_id
    support = int(mask.sum())
    correct = int((y_pred[mask] == y_true[mask]).sum())
    acc = (correct / support) if support > 0 else np.nan
    rows.append({
        "class_id": int(class_id),
        "support": support,
        "correct": correct,
        "accuracy": acc,
    })

per_class_df = pd.DataFrame(rows).sort_values("class_id").reset_index(drop=True)

# Add class metadata if available
legend_cols = [c for c in ["level_2", "LC1", "LU1", "level_1"] if c in class_table.columns]
if "level_2" in legend_cols:
    legend = class_table[legend_cols].drop_duplicates(subset=["level_2"])
    per_class_df = per_class_df.merge(legend, left_on="class_id", right_on="level_2", how="left")
    per_class_df = per_class_df.drop(columns=["level_2"])

per_class_df["accuracy_pct"] = (100 * per_class_df["accuracy"]).round(2)

print("Per-class accuracy:")
print(per_class_df.to_string(index=False))


Per-class accuracy:
 class_id  support  correct  accuracy LC1  LU1  level_1  accuracy_pct
      211    48955    44330  0.905525 B11 U111      200         90.55
      212     1178      644  0.546689 B12 U111      200         54.67
      213    22304    17810  0.798511 B13 U111      200         79.85
      214     7788     5572  0.715460 B14 U111      200         71.55
      215     6000     3853  0.642167 B15 U111      200         64.22
      216    17272    15536  0.899491 B16 U111      200         89.95
      218     4031     2000  0.496155 B18 U111      200         49.62
      219      861      528  0.613240 B19 U111      200         61.32
      221     3655     3123  0.854446 B21 U111      200         85.44
      222     5725     5184  0.905502 B22 U111      200         90.55
      223      945      770  0.814815 B23 U111      200         81.48
      230     1590     1142  0.718239 B35 U111      200         71.82
      231      494      418  0.846154 B31 U111      200         84.62


In [6]:
# Save per-class report to CSV
per_class_path = MODELS_DIR / "MLP_per_class_accuracy.csv"
per_class_df.to_csv(per_class_path, index=False)

print(f"Saved per-class accuracy report to {per_class_path}")


Saved per-class accuracy report to /workspace/eucropmap-reprod-kz/models/MLP_per_class_accuracy.csv
